# Latent Semantic Forest LM — 训练入口

在项目根目录打开本 Notebook，**选择你的 py312 CUDA 内核**，然后从上到下运行。
它直接调用项目 `train.run()`，保留原训练目标、优化器、Token 计数、tqdm 和 checkpoint 机制。

默认是 **V3 smoke 模型 + 本地真实 TinyStories 小缓存 + 512 个 CE 目标 Token**，适合先验证训练链路；
小缓存只有 32 个训练故事，不代表正式预训练。主模型和更大训练预算在下面参数区切换。

修改参数后，重新运行“配置”和“数据”单元，再启动训练。训练、评估和性能实验应顺序运行。

如果 Jupyter 没有列出 py312，可在 PowerShell 中注册这个环境，然后选择 `Python (py312 CUDA)`：

```powershell
& 'E:/anaconda3/envs/py312/python.exe' -m ipykernel install --user --name py312 --display-name "Python (py312 CUDA)"
```


In [ ]:
import copy, gc, json, os, sys, uuid
from pathlib import Path
from datetime import datetime, timezone

PROJECT_DIR = None  # 若不是从项目目录打开，可填 r"E:\dev\AI\NLP\CCG-LM"
def find_project(value=None):
    start = Path(value).expanduser().resolve() if value else Path.cwd().resolve()
    for path in [start, *start.parents]:
        if (path / "train.py").is_file() and (path / "configs").is_dir():
            return path
    raise FileNotFoundError("找不到项目根目录，请设置 PROJECT_DIR。")

PROJECT_ROOT = find_project(PROJECT_DIR)
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
from IPython.display import display
from utils import load_config, device_info, atomic_json

print("项目:", PROJECT_ROOT)
print("当前 Python:", sys.executable)
print(json.dumps(device_info(), indent=2, ensure_ascii=False))


## 1. 参数

- `CONFIG`：`configs/v3.yaml` 是 smoke；`configs/v3_main_125m.yaml` 是主模型；`configs/v3_fp32.yaml` 是全 FP32 主模型。
- `BUDGET_TOKENS` 是**累计总预算**，不是额外 Token；正式实验可自行提高。
- `EFFECTIVE_TOKENS` 是每次 optimizer update 的 CE Token 数，`MICRO_BATCH_SIZE` 影响显存和累积次数。
- `DATA_SOURCE`：`existing` 复用本地缓存；`demo` 生成合成缓存；`tinystories_subset` 下载小子集；`tinystories_full` 下载完整数据。
- 从已有检查点恢复时，使用其模型、Seed、精度、学习率和有效 Token 配置；下方 `RESUME_*` 只在显式填写时切换实现或 microbatch。


In [ ]:
CONFIG = "configs/v3.yaml"
MODE = "forest_vector4"
FOREST_IMPLEMENTATION = None  # None 跟随配置，也可选 score_first / lazy_semantic / lazy_tensorized
SEED = 42
DEVICE = "cuda"  # 使用 CPU 时显式改为 "cpu"；也可用 "auto"

BUDGET_TOKENS = 512
EFFECTIVE_TOKENS = 256
MICRO_BATCH_SIZE = 2
SAVE_EVERY_STEPS = 10
EVAL_EVERY_STEPS = 10
TRAIN_EVAL_BATCHES = 2  # 训练期开发集检查范围；不是完整开发集指标

DATA_SOURCE = "existing"
CACHE_DIR = None  # None 使用默认路径；也可指定自己的缓存目录
VERIFY_CACHE = True

RESUME_CHECKPOINT = None  # 例如 r"runs/your_run/tokens_512.pt"；只加载可信检查点
RESUME_IMPLEMENTATION = None  # 例如 "lazy_tensorized"，None 保留检查点实现
RESUME_MICRO_BATCH_SIZE = None  # OOM 后可减小；None 保持原值
RUN_DIR = None  # 新训练自动建唯一目录；恢复默认使用检查点所在目录


## 2. 生成本次配置

默认不改写仓库 YAML。解析后的配置写入本次运行目录，便于复现。
已有输出目录若含训练记录，新训练会拒绝覆盖；应改用检查点恢复或选择新目录。


In [ ]:
def project_path(value):
    p = Path(value).expanduser()
    return (p if p.is_absolute() else PROJECT_ROOT / p).resolve()

IMPLEMENTATIONS = {"reference", "score_first", "lazy_semantic", "lazy_tensorized_a", "lazy_tensorized"}
RESUME_PATH = project_path(RESUME_CHECKPOINT) if RESUME_CHECKPOINT else None
START_TOKENS = 0
if RESUME_PATH:
    if not RESUME_PATH.is_file():
        raise FileNotFoundError(RESUME_PATH)
    saved = torch.load(RESUME_PATH, map_location="cpu", weights_only=False)
    CFG = copy.deepcopy(saved["config"])
    START_TOKENS = int(saved["state"]["target_tokens"])
    if RESUME_IMPLEMENTATION is not None:
        CFG["model"]["forest_implementation"] = RESUME_IMPLEMENTATION
    if RESUME_MICRO_BATCH_SIZE is not None:
        CFG["train"]["micro_batch_size"] = RESUME_MICRO_BATCH_SIZE
    del saved
    print("恢复检查点配置；忽略新训练的 CONFIG / MODE / SEED / EFFECTIVE_TOKENS。")
else:
    CFG = load_config(project_path(CONFIG))
    CFG["model"]["mode"] = MODE
    if FOREST_IMPLEMENTATION is not None:
        CFG["model"]["forest_implementation"] = FOREST_IMPLEMENTATION
    CFG["seed"] = SEED
    CFG["train"].update(effective_tokens=EFFECTIVE_TOKENS, micro_batch_size=MICRO_BATCH_SIZE)

if CFG["model"].get("forest_implementation", "reference") not in IMPLEMENTATIONS:
    raise ValueError("未知的 CKY 实现")
if DEVICE == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("当前内核没有 CUDA：请选择 py312 CUDA 内核；要使用 CPU，请显式设置 DEVICE='cpu'。")
if min(BUDGET_TOKENS, CFG["train"]["effective_tokens"], CFG["train"]["micro_batch_size"]) <= 0:
    raise ValueError("Token 预算和 microbatch 必须为正数")
if BUDGET_TOKENS <= START_TOKENS:
    raise ValueError(f"检查点已有 {START_TOKENS} Token，请把总预算设得更大。")
if CFG.get("precision"):
    from models.precision_policy import PrecisionPolicy
    PrecisionPolicy(**CFG["precision"])  # 校验策略；实际训练入口负责设置后端
    if not hasattr(torch.backends.cuda.matmul, "fp32_precision"):
        raise RuntimeError("当前精度配置需要支持 fp32_precision 的 PyTorch；请使用本机 py312 环境。")

stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ROOT = project_path(RUN_DIR) if RUN_DIR else (
    RESUME_PATH.parent if RESUME_PATH else PROJECT_ROOT / "runs" / f"notebook_{CFG['name']}_{stamp}_{uuid.uuid4().hex[:6]}"
)
if not RESUME_PATH and RUN_ROOT.exists() and ((RUN_ROOT / "metrics.csv").exists() or any(RUN_ROOT.glob("*.pt"))):
    raise FileExistsError("该目录已有训练结果，请设置 RESUME_CHECKPOINT 或更换 RUN_DIR。")

CFG["train"].update(budget_tokens=BUDGET_TOKENS, device=DEVICE, output_dir=str(RUN_ROOT), progress=True,
                    save_every_steps=SAVE_EVERY_STEPS, eval_every_steps=EVAL_EVERY_STEPS,
                    eval_batches=TRAIN_EVAL_BATCHES, drive_dir=None)
CFG["train"]["checkpoint_tokens"] = sorted(set(
    [int(x) for x in CFG["train"]["checkpoint_tokens"] if int(x) <= BUDGET_TOKENS] + [BUDGET_TOKENS]
))
print(json.dumps({"run": str(RUN_ROOT), "resume": str(RESUME_PATH) if RESUME_PATH else None,
                  "from_tokens": START_TOKENS, "total_budget": BUDGET_TOKENS,
                  "mode": CFG["model"]["mode"], "cky": CFG["model"].get("forest_implementation"),
                  "precision": CFG.get("precision", CFG["train"]["precision"]),
                  "effective_tokens": CFG["train"]["effective_tokens"], "micro_batch_size": CFG["train"]["micro_batch_size"]},
                 indent=2, ensure_ascii=False))
DATA_READY = False


## 3. 数据准备与检查

默认优先复用 `cache/tinystories_integration`；缺失时尝试解压仓库附带的 `precision_study_cache.zip`，不需要联网。
只有显式选择 `tinystories_subset` / `tinystories_full` 才下载数据。不会静默复用规格不同的缓存。


In [ ]:
import zipfile
from data.prepare import prepare, verify_cache

sources = {"existing", "demo", "tinystories_subset", "tinystories_full"}
if DATA_SOURCE not in sources:
    raise ValueError(f"DATA_SOURCE 必须是 {sorted(sources)}")
if RESUME_PATH and DATA_SOURCE != "existing":
    raise ValueError("恢复训练请使用 existing，保持原数据指纹；缓存移动时可设置 CACHE_DIR。")

default_cache = {
    "existing": "cache/tinystories_integration", "demo": "cache/notebook_demo",
    "tinystories_subset": "cache/tinystories_integration", "tinystories_full": "cache/tinystories_full",
}
chosen_cache = CACHE_DIR or (CFG["data"]["cache_dir"] if RESUME_PATH else default_cache[DATA_SOURCE])
CACHE_ROOT = project_path(chosen_cache)
CFG["data"]["cache_dir"] = str(CACHE_ROOT)
if DATA_SOURCE == "existing":
    bundled_root = PROJECT_ROOT / "cache" / "tinystories_integration"
    bundle = PROJECT_ROOT / "precision_study_cache.zip"
    if not (CACHE_ROOT / "manifest.json").exists() and CACHE_ROOT == bundled_root and bundle.is_file():
        with zipfile.ZipFile(bundle) as z:
            for member in z.infolist():
                destination = (PROJECT_ROOT / member.filename).resolve()
                if not destination.is_relative_to(bundled_root) or member.is_dir() and not member.filename.startswith("cache/tinystories_integration"):
                    raise ValueError(f"缓存归档路径不合法: {member.filename}")
            z.extractall(PROJECT_ROOT)
    manifest_path = CACHE_ROOT / "manifest.json"
    if not manifest_path.is_file():
        raise FileNotFoundError("找不到数据缓存。请指定 CACHE_DIR，或显式选择 demo / tinystories_subset。")
    MANIFEST = json.loads(manifest_path.read_text(encoding="utf-8"))
    if VERIFY_CACHE:
        verify_cache(CACHE_ROOT, MANIFEST)
else:
    kwargs = dict(demo=DATA_SOURCE == "demo", verify=VERIFY_CACHE)
    if DATA_SOURCE == "tinystories_subset":
        kwargs.update(max_train_stories=32, max_validation_stories=8)
    MANIFEST = prepare(CFG["data"], **kwargs)

for key in ("max_history", "max_words", "max_target_length"):
    if CFG["model"][key] < MANIFEST["spec"]["data"][key]:
        raise ValueError(f"模型 {key} 小于缓存规格")
if CFG["model"]["vocab_size"] < MANIFEST["bpe_vocab_size"] or CFG["model"]["word_vocab_size"] < MANIFEST["word_vocab_size"]:
    raise ValueError("模型词表无法容纳该缓存的 token ID")
RUN_ROOT.mkdir(parents=True, exist_ok=True)
atomic_json(CFG, RUN_ROOT / "notebook_config.json")
print("数据缓存:", CACHE_ROOT, "demo=", MANIFEST["spec"]["demo"])
display(MANIFEST["stats"])
DATA_READY = True


## 4. 启动训练

此单元直接运行训练，tqdm 以真正参与 CE 的目标 Token 显示进度。
结果写入 `RUN_ROOT`：`metrics.csv/jsonl`、配置、环境、参数统计和检查点。
中断不会保存未完成的更新；恢复时选择已完整落盘的 `.pt`，而不是假设 `latest.pt` 一定最新。


In [ ]:
from train import run as train_run

if not DATA_READY:
    raise RuntimeError("请先完成数据准备单元。")
TRAINING_STATE = None
try:
    TRAINING_STATE = train_run(copy.deepcopy(CFG), str(RESUME_PATH) if RESUME_PATH else None)
except KeyboardInterrupt:
    print("训练已中断。可从本次目录中已完成的检查点恢复。")
finally:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
if TRAINING_STATE is not None:
    display(TRAINING_STATE)
print("输出目录:", RUN_ROOT)


## 5. 查看记录和检查点

下面只比较最大的 `tokens_*.pt` 与 `latest.pt` 的实际 Token 计数，避免把较旧的 latest 当成最新。


In [ ]:
import pandas as pd

def latest_complete_checkpoint(folder):
    numbered = sorted(folder.glob("tokens_*.pt"), key=lambda p: int(p.stem.split("_")[-1]))
    candidates = numbered[-1:] + ([folder / "latest.pt"] if (folder / "latest.pt").is_file() else [])
    records = []
    for path in candidates:
        saved = torch.load(path, map_location="cpu", weights_only=False)
        records.append((int(saved["state"]["target_tokens"]), int(saved["state"]["step"]), path))
        del saved
    return max(records, key=lambda x: (x[0], x[1]))[2] if records else None

LAST_CHECKPOINT = latest_complete_checkpoint(RUN_ROOT)
print("最新完整检查点:", LAST_CHECKPOINT)
if (RUN_ROOT / "metrics.csv").is_file():
    display(pd.read_csv(RUN_ROOT / "metrics.csv").tail(10))


## 6. 可选：本次运行后继续训练

默认跳过。把 `EXTRA_TOKENS` 改为正数并执行本单元，就会从最新完整检查点增加预算，保留优化器、调度、RNG 和采样游标。
另一种方式是在顶部填写 `RESUME_CHECKPOINT`，并把 `BUDGET_TOKENS` 设置为更大的累计总预算。


In [ ]:
EXTRA_TOKENS = 0
if EXTRA_TOKENS < 0:
    raise ValueError("EXTRA_TOKENS 不能为负数")
if EXTRA_TOKENS:
    if LAST_CHECKPOINT is None:
        raise FileNotFoundError("没有可恢复的完整检查点")
    saved = torch.load(LAST_CHECKPOINT, map_location="cpu", weights_only=False)
    next_cfg = copy.deepcopy(saved["config"])
    previous_tokens = int(saved["state"]["target_tokens"])
    next_cfg["train"]["budget_tokens"] = previous_tokens + EXTRA_TOKENS
    next_cfg["train"]["checkpoint_tokens"] = sorted(set(next_cfg["train"]["checkpoint_tokens"] + [previous_tokens + EXTRA_TOKENS]))
    del saved
    TRAINING_STATE = train_run(next_cfg, str(LAST_CHECKPOINT))
    CFG = next_cfg
    LAST_CHECKPOINT = latest_complete_checkpoint(RUN_ROOT)
    atomic_json(CFG, RUN_ROOT / "notebook_config.json")
    assert TRAINING_STATE["target_tokens"] == previous_tokens + EXTRA_TOKENS
    display(TRAINING_STATE)
    print("新检查点:", LAST_CHECKPOINT)
else:
    print("跳过追加训练；需要时修改 EXTRA_TOKENS。")


## 7. 学习曲线

显示真实训练日志；训练吞吐包含采样、传输、累积和完整更新，不是驻 GPU 的性能 Benchmark。


In [ ]:
import matplotlib.pyplot as plt

if (RUN_ROOT / "metrics.csv").is_file():
    metrics = pd.read_csv(RUN_ROOT / "metrics.csv")
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
    axes[0].plot(metrics.target_tokens, metrics.train_nll, "o-", label="Train")
    val = metrics.dropna(subset=["val_nll"])
    if len(val):
        axes[0].plot(val.target_tokens, val.val_nll, "o-", label="Dev subset")
    axes[0].set(xlabel="CE target tokens", ylabel="NLL (nats/token)")
    axes[0].legend()
    axes[1].plot(metrics.target_tokens, metrics.target_tokens_per_second, "o-")
    axes[1].set(xlabel="CE target tokens", ylabel="Training CE tokens/s")
    fig.tight_layout()
    fig.savefig(RUN_ROOT / "training_preview.png", dpi=150)
    plt.show()
else:
    print("尚无训练日志。")


## 8. 开发集评估

默认只评估开发集的前两个 batch，结果明确是子集指标。
把 `DEV_MAX_BATCHES=None` 可评估完整开发集；这里不自动使用 test split。


In [ ]:
RUN_DEV_EVALUATION = True
DEV_MAX_BATCHES = 2
DEV_BATCH_SIZE = 4
if RUN_DEV_EVALUATION and LAST_CHECKPOINT is not None:
    from evaluate import evaluate_checkpoints
    evaluation = evaluate_checkpoints(
        [str(LAST_CHECKPOINT)], str(CACHE_ROOT), str(RUN_ROOT / "dev_eval"),
        split="dev", batch_size=DEV_BATCH_SIZE, device_name=DEVICE,
        max_batches=DEV_MAX_BATCHES, diagnostics=False, top1=False,
    )
    display(pd.DataFrame(evaluation)[["trained_target_tokens", "target_tokens", "nll", "perplexity"]])
    print("评估范围:", "完整 dev" if DEV_MAX_BATCHES is None else f"dev 前 {DEV_MAX_BATCHES} 个 batch")
else:
    print("跳过开发集评估。")


## 输出与后续

本次目录由 `RUN_ROOT` 指定。主模型训练前，先调整参数区的 `CONFIG`、总预算与有效 batch；
正式数据需准备独立缓存，不能把默认公共小子集当作全量 TinyStories。

V3 的实现、精度和验收边界见 [实验报告](reports/cky_v3/summary.md)。
此 Notebook 不修改正式默认配置，也不自动切换检查点的模型或精度策略。
